# 06 — Artifact 1: Hyperlipidemia Condition Merge

🔧 Script · 📚 Reference

**Artifact 1:** Rhett759 has Hyperlipidemia in two sources — Synthea (SNOMED 55822004) and Epic projection (ICD-10 E78.5). These are the same condition under different code systems. Layer 3 merges them into one canonical gold-tier Condition using the UMLS-CUI bridge.


## 1. Build the two silver-tier Conditions

In [ ]:
from ehi_atlas.harmonize.provenance import SYS_SOURCE_TAG, SYS_LIFECYCLE

# Synthea: SNOMED
cond_synthea = {
    "resourceType": "Condition",
    "id": "synthea-cond-hyperlipid",
    "meta": {"tag": [
        {"system": SYS_SOURCE_TAG, "code": "synthea"},
        {"system": SYS_LIFECYCLE,  "code": "standardized"},
    ]},
    "clinicalStatus": {"coding": [{"system": "http://terminology.hl7.org/CodeSystem/condition-clinical", "code": "active"}]},
    "code": {
        "coding": [{"system": "http://snomed.info/sct", "code": "55822004", "display": "Hyperlipidemia"}],
        "text": "Hyperlipidemia",
    },
    "subject": {"reference": "Patient/rhett759"},
    "onsetDateTime": "2020-03-15",
}

# Epic projection: ICD-10 only (forces the UMLS-CUI bridge)
cond_epic = {
    "resourceType": "Condition",
    "id": "epic-cond-e785",
    "meta": {"tag": [
        {"system": SYS_SOURCE_TAG, "code": "epic-ehi"},
        {"system": SYS_LIFECYCLE,  "code": "stub-silver"},
    ]},
    "clinicalStatus": {"coding": [{"system": "http://terminology.hl7.org/CodeSystem/condition-clinical", "code": "active"}]},
    "code": {
        "coding": [{"system": "http://hl7.org/fhir/sid/icd-10-cm", "code": "E78.5", "display": "Hyperlipidemia, unspecified"}],
        "text": "Hyperlipidemia, unspecified",
    },
    "subject": {"reference": "Patient/rhett759"},
    "onsetDateTime": "2021-09-01",
}

print("Built cond_synthea (SNOMED 55822004) and cond_epic (ICD-10 E78.5)")


## 2. cluster_conditions_by_cui

In [ ]:
# 🔧 Script + 📚 Reference
from ehi_atlas.harmonize.code_map import (
    annotate_resource_codings,
    collect_concept_groups,
)

# First, annotate each condition with UMLS CUI extensions
annotate_resource_codings(cond_synthea)
annotate_resource_codings(cond_epic)

# cluster_conditions_by_cui groups by shared CUI
conditions_by_source = {
    "synthea":  [cond_synthea],
    "epic-ehi": [cond_epic],
}

clusters = collect_concept_groups([cond_synthea, cond_epic])
print(f"CUI clusters found: {len(clusters)}")
for cui, group in clusters.items():
    print(f"  CUI {cui}: {len(group)} condition(s)")
    for c in group:
        print(f"    id={c['id']}, codings={[cd['code'] for cd in c['code']['coding']]}")


## 3. merge_conditions

In [ ]:
from ehi_atlas.harmonize.condition import merge_conditions

# Take the first (and only) cluster
cui, cluster = next(iter(clusters.items()))
result = merge_conditions(cluster, f"merged-cond-rhett-hyperlipid")

merged = result.merged
print("Merged Condition ID:", merged["id"])
print("Sources:",            result.sources)
print("Rationale:",          result.rationale)


## 4. Inspect the merged result

In [ ]:
import json

print("code.coding (both systems preserved):")
for coding in merged["code"]["coding"]:
    cui_ext = next((e["valueString"] for e in coding.get("extension", [])
                    if "umls-cui" in e.get("url", "")), None)
    print(f"  {coding['system'].split('/')[-1]} {coding['code']}: {coding.get('display', '')}  →  CUI {cui_ext}")

print()
print("meta.tag (both source-tags + lifecycle=harmonized):")
for tag in merged.get("meta", {}).get("tag", []):
    print(f"  {tag.get('system','').split('/')[-1]}: {tag.get('code','')}")

print()
print("onsetDateTime (earliest = Synthea's 2020-03-15):", merged.get("onsetDateTime"))

print()
print("merge-rationale extension:")
from ehi_atlas.harmonize.provenance import EXT_MERGE_RATIONALE
for ext in merged.get("meta", {}).get("extension", []):
    if EXT_MERGE_RATIONALE in ext.get("url", ""):
        print(" ", ext.get("valueString"))


## 5. Confirm against the actual gold bundle

In [ ]:
from pathlib import Path
import json

ATLAS_ROOT = Path("..").resolve()
gold_bundle = json.loads(
    (ATLAS_ROOT / "corpus" / "gold" / "patients" / "rhett759" / "bundle.json").read_text()
)

# Find the merged Hyperlipidemia condition
gold_hyperlipid = None
for entry in gold_bundle["entry"]:
    res = entry["resource"]
    if res.get("resourceType") != "Condition":
        continue
    codings = res.get("code", {}).get("coding", [])
    codes = {c.get("code") for c in codings}
    if "55822004" in codes or "E78.5" in codes:
        gold_hyperlipid = res
        break

if gold_hyperlipid:
    print("Found in gold bundle: Condition/", gold_hyperlipid["id"])
    print("Codings:", [(c.get("system","").split("/")[-1], c["code"]) for c in gold_hyperlipid["code"]["coding"]])
    tags = [t["code"] for t in gold_hyperlipid.get("meta", {}).get("tag", [])]
    print("Source tags:", [t for t in tags if t not in ("harmonized", "gold")])
else:
    print("Not found — re-run 'make pipeline' to regenerate gold tier")


**Next:** [07_layer3_medication_artifact_2.ipynb](./07_layer3_medication_artifact_2.ipynb)